# Text-to-SVG Generation: RAG-Augmented Inference Pipeline

**Approach:** At inference time, retrieve the most similar training SVG for each test prompt and provide it as a reference for the model to modify.

## 1. Configuration & Model Loading

In [ ]:
import os
import re
import torch
import pandas as pd
import xml.etree.ElementTree as ET
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

ADAPTER_DIR = "./svg_v1/"
TRAIN_CSV_PATH = "./train_complex.csv"
EMBEDDINGS_PATH = "train_embeddings.pt"
TEST_CSV_PATH = "test.csv"
SUBMISSION_PATH = ADAPTER_DIR + "submission.csv"

BATCH_SIZE = 16
MAX_NEW_TOKENS = 2048

SYSTEM_PROMPT = (
    "You are an expert vector graphics designer. Generate highly compact, parseable, and valid SVG code. "
    "If a reference SVG is provided, intelligently modify its colors, shapes, and coordinates to match the new request. "
    "If no reference is provided, generate the SVG from scratch."
)
FORCED_HEADER = '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'

# Load RAG assets
print("Loading embeddings and training reference data...")
train_df = pd.read_csv(TRAIN_CSV_PATH)
train_embeddings = torch.load(EMBEDDINGS_PATH).to("cuda")
embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")

# Load model
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    "unsloth/Qwen2.5-Coder-3B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print("Attaching LoRA adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model = model.merge_and_unload()
model.eval()
print("Model ready for inference.")

## 2. Validation Utilities

In [ ]:
ALLOWED_TAGS = {
    'svg', 'g', 'path', 'rect', 'circle', 'ellipse', 'line', 'polyline', 
    'polygon', 'defs', 'use', 'symbol', 'clipPath', 'mask', 'linearGradient', 
    'radialGradient', 'stop', 'text', 'tspan', 'title', 'desc', 'style', 
    'pattern', 'marker', 'filter'
}

def check_validity_with_reason(svg_str):
    if not svg_str: return 0, "Empty string."
    if len(svg_str) > 16000: return 0, f"Too long ({len(svg_str)} > 16000)."
    if svg_str.count("<path") > 256: return 0, f"Too many paths ({svg_str.count('<path')} > 256)."
    try:
        root = ET.fromstring(svg_str)
        for elem in root.iter():
            tag = elem.tag.split('}')[-1]
            if tag not in ALLOWED_TAGS:
                return 0, f"Disallowed tag: <{tag}>"
    except ET.ParseError as e:
        return 0, f"XML Parse Error: {e}"
    return 1, "Valid"

def is_blank_svg(svg_str):
    if not svg_str: return True
    shape_tags = ['<path', '<rect', '<circle', '<ellipse', '<polygon', '<polyline', '<line']
    return not any(tag in svg_str for tag in shape_tags)

def fallback_svg():
    return '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"><circle cx="128" cy="128" r="64" fill="#888888"/></svg>'

## 3. RAG Retrieval + Batch Generation

For each test prompt, compute its embedding and find the most similar training SVG by cosine similarity. If the similarity exceeds the threshold, include the reference SVG in the prompt.

In [ ]:
def get_batch_references(prompts, threshold=0.5):
    """Find the best reference SVG for each prompt in a batch."""
    q_embs = embedder.encode(prompts, convert_to_tensor=True, device="cuda")
    sims = torch.nn.functional.cosine_similarity(q_embs.unsqueeze(1), train_embeddings.unsqueeze(0), dim=-1)
    
    best_indices = sims.argmax(dim=1)
    best_scores = sims.max(dim=1).values
    
    references = []
    for i, score in enumerate(best_scores):
        if score.item() > threshold:
            references.append(train_df.iloc[best_indices[i].item()]['svg'])
        else:
            references.append(None)
    return references

def generate_svg_batch(prompts, do_sample=False, temperature=None, top_p=None):
    """Generate SVGs with optional RAG reference injection."""
    references = get_batch_references(prompts)
    chat_texts = []
    
    for p, ref_svg in zip(prompts, references):
        if ref_svg:
            chat_text = (
                f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
                f"<|im_start|>user\nReference SVG:\n{ref_svg}\n\n"
                f"Modify the reference to match this new request: {p}<|im_end|>\n"
                f"<|im_start|>assistant\n{FORCED_HEADER}"
            )
        else:
            chat_text = (
                f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
                f"<|im_start|>user\n{p}<|im_end|>\n"
                f"<|im_start|>assistant\n{FORCED_HEADER}"
            )
        chat_texts.append(chat_text)

    inputs = tokenizer(chat_texts, return_tensors="pt", padding=True, truncation=True).to(model.device)
    
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=do_sample,
            temperature=temperature if do_sample else None,
            top_p=top_p if do_sample else None,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=[
                tokenizer.eos_token_id, 
                tokenizer.convert_tokens_to_ids("<|im_end|>"),
                tokenizer.encode("</svg>", add_special_tokens=False)[-1]
            ]
        )
    
    prompt_lengths = inputs['input_ids'].shape[1]
    decoded_batch = tokenizer.batch_decode(output_ids[:, prompt_lengths:], skip_special_tokens=True)
    
    results = []
    for decoded in decoded_batch:
        final_svg = FORCED_HEADER + decoded
        
        # Tag healer: close truncated SVGs
        if "</svg>" not in final_svg:
            last_open = final_svg.rfind('<')
            last_close = final_svg.rfind('>')
            if last_open > last_close: 
                unclosed = final_svg[last_open:]
                if unclosed.startswith('<path') and 'd="' in unclosed:
                    last_space = unclosed.rfind(' ')
                    if last_space > unclosed.find('d="'):
                        final_svg = final_svg[:last_open + last_space] + '" />'
                    else:
                        final_svg = final_svg + '" />'
                else:
                    final_svg = final_svg[:last_open]
            final_svg += "</svg>"
            
        results.append(final_svg)
        
    return results

## 4. Submission Loop (Two-Pass Rescue System)

Pass 1 generates with greedy decoding. Failed samples are retried in Pass 2 with temperature sampling to escape structural traps.

In [ ]:
try:
    test_df = pd.read_csv(TEST_CSV_PATH)
except FileNotFoundError:
    test_df = pd.DataFrame({"id": ["test_1", "test_2"], "prompt": ["A simple red circle", "A complex lockhole silhouette"]})

if not test_df.empty:
    final_results = {}
    failed_prompts = []
    failed_ids = []

    print(f"--- PASS 1: GREEDY RAG GENERATION ({len(test_df)} prompts) ---")
    for i in tqdm(range(0, len(test_df), BATCH_SIZE), desc="Pass 1 (Greedy)"):
        batch_df = test_df.iloc[i:i+BATCH_SIZE]
        prompts = batch_df['prompt'].tolist()
        ids = batch_df['id'].tolist()
        
        pred_svgs = generate_svg_batch(prompts, do_sample=False)
        
        for j, svg_str in enumerate(pred_svgs):
            is_valid, _ = check_validity_with_reason(svg_str)
            is_blank = is_blank_svg(svg_str)
            
            if is_valid == 1 and not is_blank:
                final_results[ids[j]] = svg_str
            else:
                failed_prompts.append(prompts[j])
                failed_ids.append(ids[j])
                final_results[ids[j]] = None

    # Pass 2: Sampling rescue
    ultimate_fail_count = 0
    
    if failed_prompts:
        print(f"\n--- PASS 2: RESCUING {len(failed_prompts)} FAILED SVGS ---")
        for i in tqdm(range(0, len(failed_prompts), BATCH_SIZE), desc="Pass 2 (Sampling)"):
            batch_prompts = failed_prompts[i:i+BATCH_SIZE]
            batch_ids = failed_ids[i:i+BATCH_SIZE]
            
            rescue_svgs = generate_svg_batch(
                batch_prompts, do_sample=True, temperature=0.4, top_p=0.95
            )
            
            for j, rescue_svg in enumerate(rescue_svgs):
                is_valid, _ = check_validity_with_reason(rescue_svg)
                is_blank = is_blank_svg(rescue_svg)
                
                if is_valid == 1 and not is_blank:
                    final_results[batch_ids[j]] = rescue_svg
                else:
                    final_results[batch_ids[j]] = fallback_svg()
                    ultimate_fail_count += 1

    submission_df = pd.DataFrame([{"id": k, "svg": v} for k, v in final_results.items()])
    submission_df.to_csv(SUBMISSION_PATH, index=False)
    
    print(f"\nTotal: {len(submission_df)} | Rescued: {len(failed_prompts) - ultimate_fail_count} | Fallbacks: {ultimate_fail_count}")
    print(f"Saved to: {SUBMISSION_PATH}")

## 5. Visualize Random Outputs

In [ ]:
from IPython.display import display, SVG, HTML
import random

NUM_SAMPLES = 10

sub_df = pd.read_csv(SUBMISSION_PATH)
try:
    test_df = pd.read_csv(TEST_CSV_PATH)
except FileNotFoundError:
    test_df = pd.DataFrame({"id": ["test_1", "test_2"], "prompt": ["A simple red circle", "A complex lockhole silhouette"]})

merged_df = pd.merge(test_df, sub_df, on="id")
sampled_df = merged_df.sample(n=min(NUM_SAMPLES, len(merged_df)))

print(f"Displaying {len(sampled_df)} random generations...\n")

for _, row in sampled_df.iterrows():
    print("=" * 60)
    print(f"ID: {row['id']}")
    print(f"Prompt: {row['prompt']}")
    print("-" * 60)
    
    svg_str = row['svg']
    if pd.isna(svg_str) or str(svg_str).strip() == "":
        print("No SVG generated")
    else:
        try:
            display(SVG(svg_str))
        except Exception as e:
            print(f"Display error: {e}")
    print()